In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q22/q22_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

### cell 2 optimized for cuDF (iter 2) ###

def_codes = ["13", "31", "23", "29", "30", "18", "17"]

# 1. extract country code and select relevant cols
customer_filtered = (
    customer[["C_ACCTBAL", "C_CUSTKEY"]]
    .assign(CNTRYCODE=customer["C_PHONE"].str.slice(0, 2))
)

# 2. keep only positive balances in the desired country codes
mask = (
    customer_filtered.C_ACCTBAL > 0.0
) & (
    customer_filtered.CNTRYCODE.isin(def_codes)
)
customer_filtered = customer_filtered[mask]

# 3. filter above-average balances
avg_value = customer_filtered.C_ACCTBAL.mean()
customer_filtered = customer_filtered[customer_filtered.C_ACCTBAL > avg_value]

# 4. find customers with no orders via an anti‐join pattern
orders_filtered = orders[["O_CUSTKEY"]].drop_duplicates()
customer_keys = customer_filtered[["C_CUSTKEY"]].drop_duplicates()

no_orders = (
    customer_keys
    .merge(
        orders_filtered,
        left_on="C_CUSTKEY",
        right_on="O_CUSTKEY",
        how="left"
    )
)
# replace .query() with boolean indexing to avoid CPU fallback
no_orders = no_orders[no_orders.O_CUSTKEY.isna()][["C_CUSTKEY"]]

# 5. re-join the filtered customers
customer_selected = (
    no_orders
    .merge(customer_filtered, on="C_CUSTKEY", how="inner")
    [["CNTRYCODE", "C_ACCTBAL"]]
)

# 6. GPU‐friendly groupby and aggregations
agg1 = (
    customer_selected
    .groupby("CNTRYCODE")
    .size()
    .reset_index(name="NUMCUST")
)
agg2 = (
    customer_selected
    .groupby("CNTRYCODE")["C_ACCTBAL"]
    .sum()
    .reset_index(name="TOTACCTBAL")
)

total = (
    agg1
    .merge(agg2, on="CNTRYCODE", how="inner")
    .sort_values("CNTRYCODE")
)